# Đo lường Hiệu năng Thực thi Thực nghiệm (Google Colab CPU-only — 25.000 bài)

Notebook này tổng hợp các chỉ số tài nguyên và thời gian thực thi thực tế từ 5 phiên chạy Milestone 2 của `TV1` – `TV5` trên **Google Colab CPU-only** (`summaries/milestone2/whole_team_summary.json`).

> **Lưu ý về phạm vi đo (Resume Scope):** Cả 5 phiên đều thực thi với `resume = true`, `storage_mode = colab-local-batches` (`batch_size = 50`, `copy_workers = 8`) và có số lượng bài đã hoàn chỉnh từ trước (`skipped_completed_papers`) khác nhau (từ `39` bài ở TV5 đến `3.586` bài ở TV3). Ngoài ra, quá trình đóng gói `MS2_RESULTS.zip` được đo tách riêng (`final_export_seconds`). Do đó, **không dùng wall time giữa các phiên để xếp hạng tốc độ giữa các thành viên**; thay vào đó cần đọc đồng thời với số bài thực sự xử lý trong phiên (`pending_papers_in_session`) và số stage migrate.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
summary_path = ROOT / 'summaries/milestone2/whole_team_summary.json'

if not summary_path.exists():
    raise FileNotFoundError(f"Không tìm thấy tệp tổng hợp MS2 tại: {summary_path}")

data = json.loads(summary_path.read_text(encoding='utf-8'))
env = data['environment']
agg_perf = data['aggregate']['performance_metrics']
members = data['members']

display(Markdown(
    f"**Nền tảng thực thi đồng nhất (5/5 thành viên):** `{env['platform']}` | "
    f"**CPU:** `{env['cpu_model']}` ({env['cpu_cores']} cores) | "
    f"**RAM hệ thống:** `{env['total_ram_gb']} GB` | **Python:** `{env['python_version']}`"
))

## 1. Thông số Phần cứng & Cấu hình 5 Phiên Thực nghiệm

Cả 5 thành viên đều chạy trên cấu hình máy ảo Google Colab CPU-only đồng nhất với cơ chế lô cục bộ (`colab-local-batches`) và đầu vào đóng băng (`frozen_input = true`):

In [ ]:
env_df = pd.DataFrame([
    {'Thông số hệ thống': 'Hệ điều hành / Kernel', 'Giá trị thực tế': env['platform']},
    {'Thông số hệ thống': 'Bộ xử lý (CPU)', 'Giá trị thực tế': f"{env['cpu_model']} ({env['cpu_cores']} vCPU)"},
    {'Thông số hệ thống': 'Tổng RAM máy ảo Colab', 'Giá trị thực tế': f"{env['total_ram_gb']} GB ({env['total_ram_gb']*1024:,.1f} MB)"},
    {'Thông số hệ thống': 'Phiên bản Python', 'Giá trị thực tế': env['python_version']},
    {'Thông số hệ thống': 'Chế độ lưu trữ & Lô', 'Giá trị thực tế': 'colab-local-batches (batch_size=50, copy_workers=8, timeout=300s/paper)'},
])
display(env_df)

session_df = pd.DataFrame([
    {
        'Thành viên': m['member'],
        'Dải ID': m['assignment_range'],
        'Hoàn tất (UTC)': m['run_timestamp_utc'][:19].replace('T', ' '),
        'Code SHA': m['code_sha'][:8],
        'Bài bỏ qua (cache)': m['skipped_completed_papers'],
        'Bài xử lý trong phiên': m['pending_papers_in_session'],
        'Stage migrate': m['migrated_stages'],
        'Timeout (>300s)': len(m['timeout_papers']),
    }
    for m in members
])
display(session_df)

## 2. Thời gian Thực thi theo Phạm vi Resume & Thời gian Đóng gói Xuất bản

- **`end_to_end_wall_time_sec`:** Thời gian khám phá, xử lý các bài còn lại (`pending`) và thu gom đầu ra trước đó (không gồm đóng gói ZIP).
- **`final_export_seconds`:** Thời gian đóng gói toàn bộ đầu ra thành tệp `MS2_RESULTS.zip` (ổn định khoảng `122 – 182 giây` cho mỗi dải 5.000 bài).
- Phiên **TV5** (`4.961` bài pending + `10.066` stage migrate từ cấu trúc cũ, `8,84 giờ`) và phiên **TV1** (`4.560` bài pending, không migrate, `4,40 giờ`) phản ánh sát nhất chi phí chạy gần toàn bộ dải 5.000 bài trên Colab CPU-only.

In [ ]:
time_rows = []
for m in members:
    pf = m['performance_metrics']
    time_rows.append({
        'Thành viên': m['member'],
        'Bài xử lý (pending)': m['pending_papers_in_session'],
        'Bài cache (skipped)': m['skipped_completed_papers'],
        'Wall time xử lý (s)': round(pf['end_to_end_wall_time_sec'], 2),
        'Wall time (giờ)': round(pf['end_to_end_wall_time_sec'] / 3600, 2),
        'Giây / bài pending': round(pf['effective_sec_per_pending_paper'], 2),
        'Đóng gói ZIP (s)': round(pf['final_export_seconds'], 2),
        'Tổng elapsed (s)': round(pf['total_elapsed_seconds'], 2),
    })

time_rows.append({
    'Thành viên': 'Tổng 5 phiên',
    'Bài xử lý (pending)': agg_perf['total_pending_papers_in_sessions'],
    'Bài cache (skipped)': agg_perf['total_skipped_completed_papers'],
    'Wall time xử lý (s)': agg_perf['sum_session_wall_time_sec'],
    'Wall time (giờ)': round(agg_perf['sum_session_wall_time_sec'] / 3600, 2),
    'Giây / bài pending': round(agg_perf['sum_session_wall_time_sec'] / agg_perf['total_pending_papers_in_sessions'], 2),
    'Đóng gói ZIP (s)': agg_perf['sum_final_export_sec'],
    'Tổng elapsed (s)': agg_perf['sum_total_elapsed_sec'],
})
time_df = pd.DataFrame(time_rows)
display(time_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
m_labels = [m['member'] for m in members]
wall_hrs = [m['performance_metrics']['end_to_end_wall_time_sec'] / 3600 for m in members]
pendings = [m['pending_papers_in_session'] for m in members]
sec_per_p = [m['performance_metrics']['effective_sec_per_pending_paper'] for m in members]

# Left: Wall time (giờ) với chú thích số bài pending
bars1 = ax1.bar(m_labels, wall_hrs, color=['#1f77b4', '#2ca02c', '#17becf', '#9467bd', '#ff7f0e'], width=0.55)
ax1.set_ylabel('Thời gian thực thi phiên (giờ)')
ax1.set_title('Wall Time Phiên Chạy & Số Bài Xử lý Thực tế (Pending)', fontweight='bold')
ax1.grid(axis='y', linestyle='--', alpha=0.6)
for idx, b in enumerate(bars1):
    h = b.get_height()
    ax1.text(b.get_x() + b.get_width()/2, h + 0.18, f"{h:.2f}h\n({pendings[idx]:,} bài)", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax1.set_ylim(0, max(wall_hrs) * 1.22)

# Right: Giây hiệu dụng / bài pending
bars2 = ax2.bar(m_labels, sec_per_p, color='#3b528b', alpha=0.88, width=0.52)
ax2.set_ylabel('Giây / bài pending')
ax2.set_title('Chi phí Thời gian Hiệu dụng trên Mỗi Bài Xử lý trong Phiên', fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.6)
for idx, b in enumerate(bars2):
    h = b.get_height()
    mig = members[idx]['migrated_stages']
    suffix = f"\n(+{mig:,} mig)" if mig > 0 else ""
    ax2.text(b.get_x() + b.get_width()/2, h + 0.12, f"{h:.2f}s{suffix}", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax2.set_ylim(0, max(sec_per_p) * 1.24)

plt.tight_layout()
plt.show()

## 3. Giám sát Bộ nhớ RAM (`psutil` RSS) trên Toàn bộ 25.000 Bài báo

Luồng giám sát nền `psutil` lấy mẫu bộ nhớ RSS trong suốt quá trình khám phá và xử lý các lô bài báo (`ram_measurement_scope = "discovery and pending processing"`). Kết quả thực nghiệm xác nhận **không có sự cố tràn bộ nhớ (OOM)** trên cả 5 phiên 5.000 bài:
- **Peak RAM cao nhất:** `3.561,79 MB` ở TV2 (chiếm **`27,45%`** tổng dung lượng `12,67 GB` RAM Colab; TV2 cũng là phiên ghi nhận bài báo nặng bị ngắt ở mốc timeout 300s).
- **Peak RAM ở 4 phiên còn lại:** nằm gọn trong khoảng `920,60 MB` – `1.970,39 MB` (`7,1%` – `15,2%` giới hạn RAM Colab).

In [ ]:
max_sys_mb = env['total_ram_gb'] * 1024
ram_rows = []
for m in members:
    pf = m['performance_metrics']
    ram_rows.append({
        'Thành viên': m['member'],
        'Avg RAM (MB)': round(pf['avg_ram_mb'], 2),
        'Peak RAM (MB)': round(pf['peak_ram_mb'], 2),
        'Tỷ lệ Peak / 12,67 GB': f"{pf['peak_ram_mb'] / max_sys_mb * 100:.2f}%",
        'Phạm vi đo RAM': m['ram_measurement_scope'],
        'Trạng thái OOM': 'Không OOM',
    })
ram_df = pd.DataFrame(ram_rows)
display(ram_df)

x = np.arange(len(members))
width = 0.36
avg_rams = [m['performance_metrics']['avg_ram_mb'] for m in members]
peak_rams = [m['performance_metrics']['peak_ram_mb'] for m in members]

fig, ax = plt.subplots(figsize=(10.5, 4.6))
b1 = ax.bar(x - width/2, avg_rams, width, label='Average RAM (MB)', color='#17becf', alpha=0.9)
b2 = ax.bar(x + width/2, peak_rams, width, label='Peak RAM (MB)', color='#ff7f0e', alpha=0.9)
ax.axhline(max_sys_mb, color='#d62728', linestyle='--', linewidth=1.6, label=f'Giới hạn Colab ({max_sys_mb:,.0f} MB / {env["total_ram_gb"]} GB)')
ax.set_xticks(x)
ax.set_xticklabels([m['member'] for m in members], fontweight='bold')
ax.set_ylabel('Bộ nhớ RSS (MB)')
ax.set_title('Mức Tiêu thụ RAM Thực tế (Avg & Peak) so với Giới hạn Google Colab CPU-only', fontweight='bold')
ax.set_ylim(0, max_sys_mb * 1.12)
ax.legend(loc='center right', frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.6)

for rect in list(b1) + list(b2):
    h = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2, h + 180, f"{h:,.0f}", ha='center', va='bottom', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Dung lượng Lưu trữ Đầu ra & Kiểm chứng Gói Xuất bản (`MS2_RESULTS.zip`)

Tổng hợp dung lượng lưu trữ của 3 bảng đầu ra chưa nén (`final_output_storage_mb`) và dung lượng 5 gói lưu trữ xuất bản `MS2_RESULTS.zip` (bao gồm các bảng dữ liệu, `occurrences.json`, `manifests/` và siêu dữ liệu kiểm chứng SHA-256):

In [ ]:
storage_rows = []
for m in members:
    pf = m['performance_metrics']
    mc = m['coverage_statistics']
    kb_per_cert = (pf['final_output_storage_mb'] * 1024) / mc['certified_complete_publications']
    storage_rows.append({
        'Thành viên': m['member'],
        'Bài đạt QA': mc['certified_complete_publications'],
        '3 bảng chưa nén (MB)': round(pf['final_output_storage_mb'], 2),
        'TB / bài đạt QA (KB)': round(kb_per_cert, 1),
        'MS2_RESULTS.zip (MB)': round(pf['archive_zip_mb'], 2),
        'SHA-256 (16 ký tự đầu)': pf['archive_sha256'][:16],
    })

tot_cert = data['aggregate']['coverage_statistics']['certified_complete_publications']
storage_rows.append({
    'Thành viên': 'Toàn nhóm',
    'Bài đạt QA': tot_cert,
    '3 bảng chưa nén (MB)': f"{agg_perf['total_final_output_storage_mb']:.2f} ({agg_perf['total_final_output_storage_mb']/1024:.2f} GB)",
    'TB / bài đạt QA (KB)': round((agg_perf['total_final_output_storage_mb'] * 1024) / tot_cert, 1),
    'MS2_RESULTS.zip (MB)': f"{agg_perf['total_archive_zip_mb']:.2f} ({agg_perf['total_archive_zip_mb']/1024:.2f} GB)",
    'SHA-256 (16 ký tự đầu)': '5/5 đã kiểm chứng',
})
storage_df = pd.DataFrame(storage_rows)
display(storage_df)

x = np.arange(len(members))
width = 0.36
out_mbs = [m['performance_metrics']['final_output_storage_mb'] for m in members]
zip_mbs = [m['performance_metrics']['archive_zip_mb'] for m in members]

fig, ax = plt.subplots(figsize=(9.8, 4.4))
b1 = ax.bar(x - width/2, out_mbs, width, label='3 bảng đầu ra chưa nén (MB)', color='#4575b4')
b2 = ax.bar(x + width/2, zip_mbs, width, label='Gói xuất bản MS2_RESULTS.zip (MB)', color='#9467bd')
ax.set_xticks(x)
ax.set_xticklabels([m['member'] for m in members], fontweight='bold')
ax.set_ylabel('Dung lượng đĩa (MB)')
ax.set_title('Dung lượng Lưu trữ Đầu ra Milestone 2 theo Thành viên (Tổng 3,68 GB bảng / 4,31 GB ZIP)', fontweight='bold')
ax.legend(frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.6)
for rect in list(b1) + list(b2):
    h = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2, h + 15, f"{h:.0f}", ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax.set_ylim(0, max(zip_mbs) * 1.18)

plt.tight_layout()
plt.show()